# Limpieza y ingeniería de atributos
Este notebook va a ser para limpiar el dataframe.

Se recibe como entrada un dataframe (puede ser el de cualquier año o el consolidado total) y se van a procesar y limpiar las siguientes columnas:

- Género: Poner únicamente los géneros en F, M, No especificado
- Edad: Verificar que no haya edades negativas o que no cuadren (muy bajas o muy altas)
- Fechas: Verificar que estén en formato DD/MM/AAAA
- Eliminar nulos y duplicados

# Bibliotecas

In [18]:
%pip install polars

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\deiss\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
import polars as pl
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import re
from collections import Counter

# Obtenemos los datos de entrada

In [20]:
df = pl.read_parquet("DATA/consolidado_2010.parquet")
df

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,str,str,str,str,str,str
"""F""","""29""","""84""","""70""","""2010-10-01""","""00:00:35.893000""","""26""","""2010-10-01""","""00:28:47.243000""","""2010/10.csv"""
"""M""","""35""","""1129""","""25""","""2010-10-01""","""00:06:51.633000""","""36""","""2010-10-01""","""00:21:07.183000""","""2010/10.csv"""
"""M""","""25""","""1086""","""7""","""2010-10-01""","""00:07:46.920000""","""80""","""2010-10-01""","""00:27:12.363000""","""2010/10.csv"""
"""M""","""28""","""1184""","""77""","""2010-10-01""","""00:08:19.213000""","""65""","""2010-10-01""","""00:33:18.540000""","""2010/10.csv"""
"""M""","""41""","""1120""","""25""","""2010-10-01""","""00:08:43.637000""","""36""","""2010-10-01""","""00:20:53.870000""","""2010/10.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""","""39""","""641""","""26""","""2010-09-29""","""23:31:59.413000""","""22""","""2010-09-30""","""00:10:01.783000""","""2010/9.csv"""
"""M""","""29""","""450""","""68""","""2010-09-29""","""23:54:44.487000""","""59""","""2010-09-30""","""00:01:20.203000""","""2010/9.csv"""
"""M""","""33""","""659""","""82""","""2010-09-29""","""23:54:45.580000""","""36""","""2010-09-30""","""00:00:55.703000""","""2010/9.csv"""


# Eliminamos nulos y duplicados

In [21]:
# Verificar cuántos nulos hay
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [22]:
# Rellenar con "No especificado" todos los nulos existentes
df = df.fill_null("No especificado")

In [23]:
# Verificamos que ya no haya nulos
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [24]:
# Ver cuántos duplicados hay
df.is_duplicated().sum()

0

In [25]:
#Para eliminar duplicados
df.unique()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,str,str,str,str,str,str
"""M""","""25""","""846""","""23""","""2010-11-03""","""12:57:14.013000""","""23""","""2010-11-03""","""13:12:10.887000""","""2010/11.csv"""
"""M""","""27""","""1132""","""13""","""2010-11-04""","""18:49:46.567000""","""20""","""2010-11-04""","""18:58:52.990000""","""2010/11.csv"""
"""M""","""28""","""1097""","""36""","""2010-04-29""","""19:40:46.333000""","""74""","""2010-04-29""","""19:47:27.323000""","""2010/4.csv"""
"""M""","""60""","""783""","""44""","""2010-05-14""","""21:11:09.247000""","""73""","""2010-05-14""","""21:21:22.083000""","""2010/5.csv"""
"""F""","""52""","""211""","""55""","""2010-12-04""","""18:39:04.177000""","""41""","""2010-12-04""","""19:04:16.637000""","""2010/12.csv"""
…,…,…,…,…,…,…,…,…,…
"""F""","""34""","""798""","""13""","""2010-04-26""","""10:23:38.053000""","""22""","""2010-04-26""","""10:25:21.273000""","""2010/4.csv"""
"""M""","""42""","""497""","""34""","""2010-08-05""","""11:31:05.570000""","""28""","""2010-08-05""","""11:37:36.803000""","""2010/8.csv"""
"""F""","""52""","""742""","""69""","""2010-10-27""","""19:32:39.467000""","""53""","""2010-10-27""","""19:40:23.887000""","""2010/10.csv"""


In [26]:
# Ver cuántos duplicados hay
df.is_duplicated().sum()

0

# Tratamiento a columna "Genero_Usuario"
Solo vamos a tomar en cuenta los registros que sean 'F', 'M' y 'No especificado'. Los registros que no sean 'F' o 'M', serán reemplazados por 'No especificado'.

Estos registros ya son conocidos, son los valores: 'O', '?' y 'nan'

In [27]:
valores_no_estandar = ['O', '?', 'nan'] 
df = df.with_columns(
    pl.col("Genero_Usuario").replace(valores_no_estandar, 'No especificado')
)

print("\nValores de 'Genero_Usuario' después de la limpieza:")
print(df['Genero_Usuario'].value_counts())


Valores de 'Genero_Usuario' después de la limpieza:
shape: (2, 2)
┌────────────────┬────────┐
│ Genero_Usuario ┆ count  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ M              ┆ 615408 │
│ F              ┆ 225671 │
└────────────────┴────────┘


# Tratamiento a columnas de fechas

In [28]:
#Revisar si podemos estandarizar los años con horas extrañas, sino, solo trabajamos desde 2017
df = df.with_columns(
    pl.col("Fecha_Retiro")
      .cast(pl.Utf8)
      .str.replace("-", "/")
      .str.strptime(pl.Date, format="%d/%m/%Y", strict=False),

    pl.col("Fecha_Arribo")
      .cast(pl.Utf8)
      .str.replace("-", "/")
      .str.strptime(pl.Date, format="%d/%m/%Y", strict=False)
)

print("2010 procesado correctamente")

2010 procesado correctamente


In [29]:
#Revisar si podemos estandarizar los años con horas extrañas, sino, solo trabajamos desde 2017
def fix_time(col):
    return (
        pl.col(col)
        # Si falta microsegundos: agregar ".000000"
        .str.replace_all(r"^(\d{1,2}:\d{2}:\d{2})$", r"\1.000000")
        # Si la hora tiene un solo dígito al inicio: anteponer "0"
        .str.replace_all(r"^(\d:)", r"0\1")
        # Convertir a time
        .str.strptime(pl.Time, format="%H:%M:%S%.f", strict=False)
    )

df = df.with_columns(
    [
        fix_time("Hora_Retiro").alias("Hora_Retiro"),
        fix_time("Hora_Arribo").alias("Hora_Arribo"),
    ]
)

df

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,date,time,str,date,time,str
"""F""","""29""","""84""","""70""",null,00:00:35.893,"""26""",null,00:28:47.243,"""2010/10.csv"""
"""M""","""35""","""1129""","""25""",null,00:06:51.633,"""36""",null,00:21:07.183,"""2010/10.csv"""
"""M""","""25""","""1086""","""7""",null,00:07:46.920,"""80""",null,00:27:12.363,"""2010/10.csv"""
"""M""","""28""","""1184""","""77""",null,00:08:19.213,"""65""",null,00:33:18.540,"""2010/10.csv"""
"""M""","""41""","""1120""","""25""",null,00:08:43.637,"""36""",null,00:20:53.870,"""2010/10.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""","""39""","""641""","""26""",null,23:31:59.413,"""22""",null,00:10:01.783,"""2010/9.csv"""
"""M""","""29""","""450""","""68""",null,23:54:44.487,"""59""",null,00:01:20.203,"""2010/9.csv"""
"""M""","""33""","""659""","""82""",null,23:54:45.580,"""36""",null,00:00:55.703,"""2010/9.csv"""


In [31]:
# =============================================================================
#    CONFIGURACIÓN INICIAL Y ANÁLISIS EXPLORATORIO
# =============================================================================

print("=== ANÁLISIS INICIAL DE EDAD_USUARIO ===")

# Primero convertimos a string y limpiamos espacios
df = df.with_columns([
    pl.col("Edad_Usuario")
    .cast(pl.Utf8)
    .str.strip_chars()  # Limpiamos espacios en blanco
    .alias("Edad_Usuario")
])

# Estadísticas iniciales de la columna
print(f"Total de registros: {len(df):,}")
print(f"Valores nulos: {df['Edad_Usuario'].null_count()}")
print(f"Valores únicos: {df['Edad_Usuario'].n_unique()}")

# Mostrar algunos valores únicos para verificar formato
print(f"\nPrimeros 10 valores únicos: {df['Edad_Usuario'].unique()[:10].to_list()}")

=== ANÁLISIS INICIAL DE EDAD_USUARIO ===
Total de registros: 841,079
Valores nulos: 0
Valores únicos: 65

Primeros 10 valores únicos: ['79', '61', '42', '19', '63', '22', '76', '69', '31', '67']


In [32]:
# =============================================================================
# CONVERSIÓN A NUMÉRICO Y DETECCIÓN DE OUTLIERS
# =============================================================================

# Convertir a numérico (manejando errores de conversión)
df = df.with_columns([
    pl.col("Edad_Usuario")
    .cast(pl.Int64, strict=False)  # strict=False permite manejar valores no convertibles
    .alias("Edad_Usuario_Numeric")
])

print("=== ANÁLISIS DE VALORES ATÍPICOS ===")
print(f"Valores nulos después de conversión: {df['Edad_Usuario_Numeric'].null_count()}")

# Contar outliers según criterios establecidos (menores de 16, mayores de 70)
menores_16 = df.filter(pl.col("Edad_Usuario_Numeric") < 16).height
mayores_70 = df.filter(pl.col("Edad_Usuario_Numeric") > 70).height
nulos = df['Edad_Usuario_Numeric'].null_count()

print(f"Edades menores de 16 años: {menores_16:,}")
print(f"Edades mayores de 70 años: {mayores_70:,}")
print(f"Valores nulos: {nulos:,}")
print(f"Total de outliers a tratar: {menores_16 + mayores_70 + nulos:,}")

=== ANÁLISIS DE VALORES ATÍPICOS ===
Valores nulos después de conversión: 0
Edades menores de 16 años: 0
Edades mayores de 70 años: 952
Valores nulos: 0
Total de outliers a tratar: 952


In [33]:
# =============================================================================
# CÁLCULO DE MEDIANA Y REEMPLAZO DE OUTLIERS
# =============================================================================

# Calcular mediana excluyendo outliers para obtener un valor representativo
edades_validas = df.filter(
    (pl.col("Edad_Usuario_Numeric") >= 16) & 
    (pl.col("Edad_Usuario_Numeric") <= 70) &
    (pl.col("Edad_Usuario_Numeric").is_not_null())
)

mediana = edades_validas.select(pl.col("Edad_Usuario_Numeric").median()).item()
print(f"Mediana calculada (solo edades 16-70 años): {mediana}")

# Reemplazar outliers y nulos con la mediana calculada
df = df.with_columns([
    pl.when(
        (pl.col("Edad_Usuario_Numeric") < 16) | 
        (pl.col("Edad_Usuario_Numeric") > 70) |
        (pl.col("Edad_Usuario_Numeric").is_null())
    )
    .then(pl.lit(int(mediana)))  # Asignar mediana a outliers
    .otherwise(pl.col("Edad_Usuario_Numeric"))  # Mantener valores válidos
    .cast(pl.Int64)  # Asegurar tipo entero
    .alias("Edad_Usuario_Limpio")  # Nueva columna limpia
])

Mediana calculada (solo edades 16-70 años): 32.0


In [34]:
# =============================================================================
# VERIFICACIÓN Y VALIDACIÓN DE RESULTADOS
# =============================================================================

print("=== VERIFICACIÓN FINAL ===")
print(f"Valores nulos después de limpieza: {df['Edad_Usuario_Limpio'].null_count()}")
print(f"Nueva edad mínima: {df['Edad_Usuario_Limpio'].min()}")
print(f"Nueva edad máxima: {df['Edad_Usuario_Limpio'].max()}")
print(f"Mediana final: {df['Edad_Usuario_Limpio'].median()}")

# Distribución completa de edades limpias
print("\n=== DISTRIBUCIÓN DE EDADES LIMPIAS ===")
print(df['Edad_Usuario_Limpio'].describe())

=== VERIFICACIÓN FINAL ===
Valores nulos después de limpieza: 0
Nueva edad mínima: 16
Nueva edad máxima: 70
Mediana final: 32.0

=== DISTRIBUCIÓN DE EDADES LIMPIAS ===
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 841079.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 34.183419 │
│ std        ┆ 9.978796  │
│ min        ┆ 16.0      │
│ 25%        ┆ 27.0      │
│ 50%        ┆ 32.0      │
│ 75%        ┆ 40.0      │
│ max        ┆ 70.0      │
└────────────┴───────────┘


In [35]:
# =============================================================================
# EJEMPLOS DE LA TRANSFORMACIÓN
# =============================================================================

# Mostrar ejemplos de antes y después para verificar la transformación
print("=== EJEMPLOS DE LIMPIEZA ===")
ejemplos = df.select([
    "Edad_Usuario", 
    "Edad_Usuario_Limpio"
]).head(10)
print(ejemplos)

print("\n ¡Limpieza completada! Columna 'Edad_Usuario_Limpio' creada con tipo Int64")
print("Sin valores nulos ni outliers")
print("Lista para análisis estadístico")

=== EJEMPLOS DE LIMPIEZA ===
shape: (10, 2)
┌──────────────┬─────────────────────┐
│ Edad_Usuario ┆ Edad_Usuario_Limpio │
│ ---          ┆ ---                 │
│ str          ┆ i64                 │
╞══════════════╪═════════════════════╡
│ 29           ┆ 29                  │
│ 35           ┆ 35                  │
│ 25           ┆ 25                  │
│ 28           ┆ 28                  │
│ 41           ┆ 41                  │
│ 30           ┆ 30                  │
│ 34           ┆ 34                  │
│ 31           ┆ 31                  │
│ 31           ┆ 31                  │
│ 47           ┆ 47                  │
└──────────────┴─────────────────────┘

 ¡Limpieza completada! Columna 'Edad_Usuario_Limpio' creada con tipo Int64
Sin valores nulos ni outliers
Lista para análisis estadístico
